BERT

In [1]:
!pip install gensim
!pip install tensorflow
!pip install transformers==4.46.3
from google.colab import files
uploaded=files.upload()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 25.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 29.7 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.20.1
    Uninstalling huggingface_hub-1.20.1:
      Successfully uninstalled huggingface_hub-1.20.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.12.1
    Uninstalling transformers-5.12.1:
      Successfully uninstalled transformers-5.12.1
ERROR: pip's dependency resolver does not currently take into a

Saving data_mr.csv to data_mr.csv


In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Embedding, Conv1D,LSTM
from tensorflow.keras.layers import MaxPooling1D
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense,Flatten, Input, Dropout
from sklearn.model_selection import train_test_split
from gensim.parsing.preprocessing import preprocess_string
from nltk import sent_tokenize
from transformers import (BertTokenizer,
                          TFBertForSequenceClassification
)




In [3]:
df = pd.read_csv('data_mr.csv').iloc[5000:20000]

In [4]:
df.drop_duplicates(inplace=True)


In [5]:
df

,text,label
5000,When I typed Savage Intruder into the IMDb's s...,0
5001,Every once in awhile I'll remember that I've a...,0
5002,"From the brilliant mind that brought us ""The E...",0
5003,Man this thing bites! I am sorry I ever sat do...,0
5004,When I first heared that there was going to be...,0
...,...,...
19995,Very violent and nicely filmed movie. Paulina ...,1
19996,"Is Thursday an original film? Heck no, but it ...",1
19997,I loved this film! Fantastically original and ...,1
19998,This movie was excellent from start-to-finish....,1


In [6]:
X = df["text"].astype(str)
y = df["label"]

In [7]:
df

,text,label
5000,When I typed Savage Intruder into the IMDb's s...,0
5001,Every once in awhile I'll remember that I've a...,0
5002,"From the brilliant mind that brought us ""The E...",0
5003,Man this thing bites! I am sorry I ever sat do...,0
5004,When I first heared that there was going to be...,0
...,...,...
19995,Very violent and nicely filmed movie. Paulina ...,1
19996,"Is Thursday an original film? Heck no, but it ...",1
19997,I loved this film! Fantastically original and ...,1
19998,This movie was excellent from start-to-finish....,1


In [8]:
X_train, X_test, y_train, y_test=train_test_split(X,y,test_size=0.2,random_state=7)

In [9]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

train_encodings = tokenizer(
    X_train.tolist(),
    truncation=True,
    padding="max_length",
    max_length=256,
    return_tensors="tf"
)

test_encodings = tokenizer(
    X_test.tolist(),
    truncation=True,
    padding="max_length",
    max_length=256,
    return_tensors="tf"
)
train_encodings["input_ids"]
train_encodings["attention_mask"]

test_encodings["input_ids"]
test_encodings["attention_mask"]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

<tf.Tensor: shape=(2989, 256), dtype=int32, numpy=
array([[1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 1, 1, 1],
       [1, 1, 1, ..., 1, 1, 1],
       ...,
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 1, 1, 1]], dtype=int32)>

In [10]:
'''model=Sequential()
model.add(Embedding(input_dim=max_words,output_dim=128,input_length=max_len))
model.add(LSTM(128,dropout=0.2,return_sequences=True))
model.add(LSTM(64,dropout=0.2))
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(16, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(8, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(1, activation='sigmoid'))
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)'''

"model=Sequential()\nmodel.add(Embedding(input_dim=max_words,output_dim=128,input_length=max_len))\nmodel.add(LSTM(128,dropout=0.2,return_sequences=True))\nmodel.add(LSTM(64,dropout=0.2))\nmodel.add(Dense(32, activation='relu'))\nmodel.add(Dropout(0.2))\nmodel.add(Dense(16, activation='relu'))\nmodel.add(Dropout(0.2))\nmodel.add(Dense(8, activation='relu'))\nmodel.add(Dropout(0.2))\nmodel.add(Dense(1, activation='sigmoid'))\nmodel.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])\nearly_stop = EarlyStopping(\n    monitor='val_loss',\n    patience=5,\n    restore_best_weights=True\n)"

In [16]:
import tensorflow as tf
import tf_keras as keras

from transformers import TFBertForSequenceClassification

# Create tf.data datasets
train_dataset = tf.data.Dataset.from_tensor_slices(
    (
        {
            "input_ids": train_encodings["input_ids"],
            "attention_mask": train_encodings["attention_mask"],
        },
        y_train.values,
    )
).shuffle(1000).batch(32)

test_dataset = tf.data.Dataset.from_tensor_slices(
    (
        {
            "input_ids": test_encodings["input_ids"],
            "attention_mask": test_encodings["attention_mask"],
        },
        y_test.values,
    )
).batch(32)


model = TFBertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2,
)


from transformers import create_optimizer

optimizer, schedule = create_optimizer(
    init_lr=2e-5,
    num_train_steps=len(train_dataset) * 4,
    num_warmup_steps=int(0.1 * len(train_dataset) * 4),
    weight_decay_rate=0.01
)

loss = keras.losses.SparseCategoricalCrossentropy(
    from_logits=True
)
import tf_keras as keras

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

model.compile(
    optimizer=optimizer,
    loss=loss,
    metrics=["accuracy"],

)

All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [17]:
model.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=5,
    callbacks=[early_stop]
)

Epoch 1/5
374/374 [==============================] - 826s 2s/step - loss: 0.3688 - accuracy: 0.8263 - val_loss: 0.2224 - val_accuracy: 0.9100
Epoch 2/5
374/374 [==============================] - 800s 2s/step - loss: 0.1613 - accuracy: 0.9414 - val_loss: 0.2308 - val_accuracy: 0.9190
Epoch 3/5
374/374 [==============================] - 799s 2s/step - loss: 0.0821 - accuracy: 0.9727 - val_loss: 0.2735 - val_accuracy: 0.9164


In [18]:
predictions = model.predict(test_dataset)

logits = predictions.logits

import tensorflow as tf

y_pred = tf.argmax(logits, axis=1).numpy()

94/94 [==============================] - 67s 686ms/step


In [19]:
from sklearn.metrics import accuracy_score, f1_score

print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

Accuracy: 0.9100033456005353
F1 Score: 0.9134770022515278
